# Realizando Inspección al DataFrame en General

In [ ]:
import pandas as pd
import os
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# # Configuración de las rutas
PATH = Path().cwd()
PATH = PATH.parent if PATH.name == "notebooks" else PATH
DATA = 'data'
RAW_DATA = 'raw/full_data.csv'
PROCESSED_DATA = 'processed/clean_data.csv'

print(f'La nueva ruta de los archivos ha sido asignado a: {PATH}')

In [ ]:
df = pd.read_csv(PATH / DATA / RAW_DATA)

df.head()

No hay valores faltantes en el conjunto de datos.

In [ ]:
#| include: False

sum(df.isna().sum())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

**Figura X:** *Analizando valores faltantes que pueden estar presente en el conjunto de datos*

In [ ]:
sns.heatmap(df.isna(), cbar=False)
plt.show()

Este caso se van a revisar el número de variable que hay por tipo de formato, en este caso:

+ Númericas (Discreta o Continua)

+ Cualitativas (Ordinal o Nominal)

In [ ]:
X_cat = df.select_dtypes('object')
X_num = df.select_dtypes('number')

Se tienen 244 variables númericas y 19 variables categoricas.

In [ ]:
X_num.shape[1], X_cat.shape[1]

## Revisión de Variables Categóricas

In [ ]:
X_cat.columns

In [ ]:
X_cat.iloc[:, :10].head()

Separando fechas como númericas:

In [ ]:
# Separar los componentes

df[["year", "week", "canton"]] = df["week_canton"].str.split("-", expand=True)

In [ ]:
# Convertir a enteros

df["year"] = df["year"].astype(int)
df["week"] = df["week"].astype(int)

In [ ]:
# Crear una fecha a partir de año + semana (usando el lunes de esa semana ISO)

df["date"] = df.apply(lambda x: pd.to_datetime(f'{x.year}-W{x.week}-1', format='%G-W%V-%u'), axis=1)

In [ ]:
# Tirando la columna que no es de utilidad

df.drop('week_canton', axis=1, inplace=True)

+ Observando la composición de las primeras columnas categóricas

En este caso se analizando la composición de cada variable y sus cantidades únicas.

In [ ]:
col_cat = X_cat.drop('week_canton', axis=1).columns
resultados = {}

for col in col_cat:
    resultados[col] = X_cat[col].unique()

df_resumen_cat = pd.DataFrame(dict([(k, pd.Series(v)) for k,v in resultados.items()]))

Se observan que todos los niveles están correctamente asignados, en este caso en los niveles de:

+ Bajo
+ Medio
+ Alto

In [ ]:
df_resumen_cat

In [ ]:
transform_cat_to_numeric = {
    'Bajo' : 0,
    'Medio' : 1,
    'Alto' : 2
}

transform_num_to_cat = {
    0 : 'Bajo',
    1 : 'Medio',
    2 : 'Alto'
}

In [ ]:
for col in df_resumen_cat.columns:
    df[col] = df[col].map(transform_cat_to_numeric)
    print(f'Proceso realizado a: {col}')

In [ ]:
df.to_csv(PATH / DATA / PROCESSED_DATA, index=False)

print(f'Los datos han sido guardado en: {PATH / DATA / PROCESSED_DATA}')

In [ ]:
del df, X_cat, X_num, df_resumen_cat, transform_cat_to_numeric, transform_num_to_cat

Proceso finalizado de limpieza superficial.

# Efectvidad de las compañas 

> Análisis con Marianne

In [ ]:
df_camp = pd.read_csv(PATH / DATA / 'raw' / 'campaigns_eda.csv')

df_camp.head()

In [ ]:
df_camp.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
df_camp.tail()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
sns.heatmap(df_camp.isna(), cbar=False)
plt.show()

In [ ]:
df_camp[df_camp['ID'].isna()]

In [ ]:
df_camp[(df_camp['ID'].notna()) & (df_camp['canton'] == 'San Jose')]